# Missing value reliability checks

Our goal is reliable AFIB classification with incomplete ECG measurements. This notebook checks whether the numerical filling pipeline behaves correctly and measures recovery of deliberately hidden values. **It does not train classifiers or establish medical benefit.**

- Compare training medians with a fixed ridge regression filler.
- Learn 216 continuous non-P targets; keep 96 P-related numerical fields, detection counts and morphology on the existing baseline rules.
- Preserve measured values and the original missingness indicators.
- Fit only on one existing inner-training partition; use 256 inner-validation ECGs for this engineering check. No outer-test ECG enters the check.

Run `01_preprocessing.ipynb` first. This optional experiment reads only `processed_data/` and saves audit results in `reports/imputation/`; it does not change the model-input folder.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT = Path.cwd().resolve()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
assert (PROJECT / 'src/afib/imputation.py').is_file()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from scripts.audit_imputation import run_audit


## Fixed protocol

For each eligible target, ridge uses at most eight other continuous predictors selected by training-only absolute correlation, a ridge penalty of 1 and at least 20 target/joint observations. Labels, metadata, Global fields, counts, categories and the target itself are excluded as predictor inputs. Predictions use the original available fields once; generated values are not fed back into other regressions.

If support is insufficient, no predictor is eligible, or all selected predictors are absent, use the training median and audit the reason. Entirely missing training targets use a flagged zero fallback. Generated negative durations/IQRs are clipped and audited, and QT/QRS relationships are reported without claiming that clipping makes estimates physiological.

Hide nominally 10% of observed continuous cells using independent draws, or all fields from 1, 3 or 6 leads. Repeat each mask three times with paired draws. Score only originally observed eligible targets; natural gaps have no known recovery truth. Compare observed inputs and three missing leads as the main future AFIB conditions; the other masks describe severity.


In [ ]:
audit = run_audit(
    PROJECT / 'reports/imputation', validation_limit=256,
    processed_dir=PROJECT / 'processed_data',
)
assert audit['outer_test_ecgs_used'] == 0
assert audit['eligible_continuous_targets'] == 216
assert audit['fold_checks']['outer_folds'] == 10
assert audit['fold_checks']['inner_folds_per_outer'] == 3
assert audit['classifiers_trained'] is False
assert audit['clinical_benefit_established'] is False
print({key: audit[key] for key in ['scope', 'train_ecgs', 'validation_ecgs',
                                   'validation_subgroups', 'n_method_scenario_repeat_checks',
                                   'course_status']})


## Inspect recovery without confusing it with diagnosis

Per-feature MAE retains the source measurement units. The summary below divides errors by positive training IQRs before averaging; it does not mix raw errors measured in different units. Inspect `recovery_by_feature.csv`, coverage, unsupported targets, fallback counts and `recovery_by_subgroup.csv` before interpreting a difference. These 256 ECGs are an implementation sample, not a complete nested experiment or independent clinical validation set.


In [ ]:
report = PROJECT / 'reports/imputation'
recovery = pd.read_csv(report / 'recovery_summary.csv')
features = pd.read_csv(report / 'recovery_by_feature.csv')
subgroups = pd.read_csv(report / 'recovery_by_subgroup.csv')
mask_doses = pd.read_csv(report / 'mask_audits.csv')
assert 'feature' in features.columns
assert set(recovery.method) == {'median', 'ridge'}
assert len(recovery) == 24  # 2 fillers, 4 mask conditions, 3 repeats.
assert recovery.coverage.eq(1).all()
assert recovery.n_unresolved.eq(0).all()
assert len(mask_doses) == 12
assert mask_doses.achieved_continuous_mask_fraction.between(0, 1).all()
paired_counts = recovery.pivot(index=['scenario', 'repeat'], columns='method', values='n_targets')
assert paired_counts['median'].equals(paired_counts['ridge'])
display(recovery[['method', 'scenario', 'repeat', 'n_targets', 'coverage',
                  'cell_mean_mae_in_training_iqr']])
display(subgroups.groupby(['subgroup', 'method']).agg(
    n_ecgs=('n_ecgs', 'first'), checks=('scenario', 'size')))
display(mask_doses[['scenario', 'repeat', 'achieved_continuous_mask_fraction']])
print('All recovery and pipeline checks passed. No AFIB performance claim is made.')


## The next evidence needed

Train median and ridge versions of original-feature logistic regression with the same manual 10 outer / 3 inner patient folds, three C values, class weights and selection rules. Keep each branch's imputer, scaling and threshold fixed on held-out ECGs. Use a separate imputation prediction file so these extra runs do not masquerade as the main A-D design.

Macro-F1 and AFIB sensitivity are the primary prediction outcomes. Report specificity, false alarms and unweighted Brier/reliability alongside absolute scores and paired changes. Better recovery alone does not meet the AFIB goal. Any calibration uses inner out-of-fold predictions only. Threshold 0.5 is primary; the optional inner-selected 0.90 sensitivity target is a project choice and must be tested, not assumed, on outer patients.

Average repeats within each outer fold, then show mean and sample-SD error bars across folds. Report trade-offs or inconclusive results rather than inventing a clinical acceptance margin. The course also requires all three main classifiers, middle-setting selection evidence and at least three manually calculated final results with error bars. This engineering check does not complete those requirements.
